## **PORTAFOLIO ACADÉMICO DE BI:**
# **PROYECTO INMOBILIARIO — CIUDAD DE LA COSTA**
# **Análisis de Negocio — Rentabilidad e Inversión**
---
## Módulo 2:
* Introducción
* Conexión de Vista del Módulo 1
* Preguntas empresariales:

1. Análisis de rentabilidad y precios por m2

2. Rentabilidad: venta vs. alquiler

3. Terreno vs. vivienda construida

4. Mix ideal de portafolio

5. Dinámica de monedas

6. ¿El tamaño de la propiedad afecta el precio por m2?

---
# **Introducción**
## **Del Estudio Académico al Análisis Empresarial:**
En el Módulo 1 construimos y validamos la infraestructura de datos del proyecto, establecimos la conexión segura a nuestro Data Warehouse en BigQuery, transformamos los tipos de datos originales mediante SQL y evaluamos la confiabilidad estadística de la muestra por zona.
El objetivo fue mostrar paso a paso cómo se construye y valida una base de datos apta para análisis, para que cualquier lector pueda auditar de dónde salen los números antes de confiar en ellos.

A partir del Módulo 2 el enfoque cambia: pasamos de preparar la información a utilizarla para responder preguntas de negocio concretas que un inversor, una inmobiliaria o un desarrollador se haría antes de tomar una decisión de capital.

Cada Módulo de problemas empresariales va a partir de un bloque temático de preguntas resueltas mediante consultas SQL sobre la Vista (v_ciudad_de_la_costa ya consolidada en el Módulo 1) con una interpretación comercial de los resultados.

**Todos los hallazgos de este Módulo siguen sujetos a los mismos límites metodológicos ya documentados por lo que cada conclusión de negocio debe leerse con ese contexto de fondo.**

---

### **Conexión y recuperación de Datos del Módulo 1:**

Para dar inicio al análisis vinculamos el nuevo cuaderno de trabajo con la infraestructura de datos que ya dejamos consolidada en el módulo anterior ejecutando dos celdas clave para reestablecer esta conexión de forma segura:

1. **Autenticación de la Cuenta:** En la primera celda, validamos nuestras credenciales de Google para autorizar el acceso desde este nuevo entorno de Colab.
2. **Inicialización del Cliente y Carga de Datos:** En la segunda celda, nos conectamos formalmente a nuestro proyecto de BigQuery (`proyectosuy`) para llamar directamente a la Vista unificada que construimos y limpiamos previamente, asegurando que trabajaremos exactamente con el mismo set de datos validado.

In [ ]:
# Autenticamos la cuenta de Google en el nuevo cuaderno
from google.colab import auth
auth.authenticate_user()

print("Autenticación completada con éxito")

¡Autenticación completada con éxito!


In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd

# 1. Nos autenticamos asegurando el proyecto destino
auth.authenticate_user(project_id="proyectosuy")

# 2. Inicializamos el cliente pasándole explícitamente el ID en el constructor
client = bigquery.Client(project="proyectosuy")

# 3. Definimos la consulta
query_toda_la_base = """
    SELECT *
    FROM `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
"""

# 4. Cargamos todo el DataFrame completo
# Pasamos el project_id también en el método de ejecución por seguridad
df_completo = client.query(query_toda_la_base, project="proyectosuy").to_dataframe()

# Verificamos que cargaron todas las columnas y filas
print(f"Base de datos cargada. Total de propiedades: {len(df_completo)}")
df_completo.head()

¡Por fin! Base de datos cargada. Total de propiedades: 555


,id,publicacion,finalizacion,operacion,tipo_inmueble,moneda,precio,ubicacion,zona,pisos,...,banos,cochera,parrillero,jardin,piscina,mts2_terreno,mts2_edificado,antiguedad,gastos_comunes_UYU,detalles
0,1,2025-06-01,NaT,Venta,Casa,U$S,289000.0,Solymar,Sur,2,...,4,1,True,True,False,364,139.0,20,NaN,Construcción sólida
1,2,2026-05-07,NaT,Venta,Casa,U$S,250000.0,Solymar,Sur,1,...,2,1,True,True,False,310,95.0,1,NaN,Construcción sólida
2,3,2026-05-05,NaT,Venta,Complejo,U$S,187000.0,Solymar,Norte,2,...,2,2,True,False,False,148,78.2,0,NaN,Próximo a Car One
3,4,2026-01-12,NaT,Venta,Casa,U$S,265000.0,Solymar,Sur,1,...,2,1,True,True,False,527,210.0,30,NaN,Casa independiente
4,5,2026-04-02,NaT,Venta,Casa,U$S,185000.0,Lagomar,Norte,1,...,1,1,True,True,False,200,85.0,1,NaN,Próximo a Almenara Mall


# **Preguntas empresariales:**

### **Pregunta 1. Análisis de rentabilidad y precios por m2**

El primer análisis de negocio consiste en evaluar el comportamiento del precio por metro cuadrado en las distintas zonas de Ciudad de la Costa para comparar el valor real del espacio construido independientemente del tamaño total de las propiedades.


In [ ]:
# =============================================================================
# PRECIO POR M2 EDIFICADO POR BARRIO (VIVIENDAS EN VENTA, USD)
# =============================================================================

# Para evitar que una sola propiedad grande distorsione el promedio de todo
# el barrio, primero calculamos el $/m² de CADA propiedad individualmente
# (subconsulta CTE), y recién después promediamos esos valores por barrio.
query_precio_m2 = """
WITH precio_m2_individual AS (
    SELECT
        ubicacion AS Barrio,
        precio,
        mts2_edificado,

        -- Dividimos el precio de cada propiedad por sus metros edificados.
        -- Usamos SAFE_DIVIDE para que la consulta no falle si algún registro
        -- tuviera 0 en metros edificados (división por cero).
        SAFE_DIVIDE(precio, mts2_edificado) AS precio_por_m2
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        -- Excluimos terrenos: un terreno no tiene "m2 edificado", el campo no aplica.
        LOWER(tipo_inmueble) != 'terreno'

        -- Trabajamos solo en dólares, para no mezclar magnitudes de distintas monedas.
        AND LOWER(moneda) = 'u$s'

        -- Excluimos alquileres: mezclar precio de venta con precio de alquiler
        -- en un mismo $/m² no tiene sentido económico (son escalas distintas).
        AND LOWER(operacion) = 'venta'

        -- Filtramos los registros que no tienen dato de metros edificados,
        -- ya que sin ese valor no se puede calcular el precio por m2.
        AND mts2_edificado IS NOT NULL
        AND mts2_edificado > 0
)

# Con el $/m2 ya calculado por propiedad, ahora agregamos por barrio
# para obtener las métricas de resumen que nos interesan comparar.
SELECT
    Barrio,

    -- Cantidad de propiedades que efectivamente entraron en el cálculo
    -- (después de aplicar los filtros de arriba). Es distinto al total
    -- de viviendas del barrio, porque descarta las que no tienen m2 edificado.
    COUNT(*) AS Cantidad_Propiedades_Consideradas,

    -- Promedio del $/m2 individual de cada propiedad del barrio.
    ROUND(AVG(precio_por_m2), 0) AS Precio_Promedio_x_M2_USD,

    -- Mediana del $/m2: el valor central, poco afectado por outliers.
    -- APPROX_QUANTILES con el parámetro 2 divide los datos en 2 partes iguales,
    -- y OFFSET(1) toma el punto de corte del medio (equivalente al percentil 50).
    ROUND(APPROX_QUANTILES(precio_por_m2, 2)[OFFSET(1)], 0) AS Precio_Mediano_x_M2_USD,

    -- Valores extremos, para dimensionar el rango de precios del barrio.
    ROUND(MIN(precio_por_m2), 0) AS Precio_Minimo_x_M2_USD,
    ROUND(MAX(precio_por_m2), 0) AS Precio_Maximo_x_M2_USD
FROM
    precio_m2_individual
GROUP BY
    Barrio
ORDER BY
    Precio_Promedio_x_M2_USD DESC;
"""

# Ejecutamos la consulta en BigQuery y cargamos el resultado en un DataFrame de Pandas.
df_precio_m2 = client.query(query_precio_m2).to_dataframe()

# Desplegamos la tabla final con el diagnóstico de precio por m2 de cada barrio.
df_precio_m2

,Barrio,Cantidad_Propiedades_Consideradas,Precio_Promedio_x_M2_USD,Precio_Mediano_x_M2_USD,Precio_Minimo_x_M2_USD,Precio_Maximo_x_M2_USD
0,Carrasco,19,2724.0,3000.0,611.0,5009.0
1,Lagomar,25,2641.0,2846.0,1350.0,3409.0
2,Tahona,28,2553.0,2700.0,831.0,4160.0
3,Shangrila,30,2465.0,2722.0,694.0,4091.0
4,Solymar,167,2314.0,2396.0,654.0,4205.0
5,Pinar,15,1666.0,1286.0,580.0,3636.0


### **Diagnóstico: Precio por m2 edificado por barrio**

En el Módulo 1 La Tahona era el barrio más caro en precio absoluto de vivienda (USD 650,033 en promedio).

Ahora en cambio, ocupa el tercer lugar en USD/m2 (USD 2,553) detrás de Carrasco (USD 2,724) y Lagomar (USD 2,641), esto tiene una explicación de negocio muy clara: las viviendas de La Tahona son mucho más grandes, así que su precio absoluto alto no viene de un metro cuadrado carísimo, sino de que cada casa tiene muchos más metros: pagar más caro en total no siempre significa pagar más caro por metro.

*   **1. Carrasco lidera el ranking de USD/m2 pero con muestra reducida:** Con USD 2,724 en promedio (y una mediana aún más alta de USD 3,000), Carrasco es el barrio más caro por metro cuadrado de toda Ciudad de la Costa pero la base es de solo 19 propiedades (la más chica de la tabla junto con el Pinar), por lo que esta cifra debe tomarse como una referencia orientativa más que como un dato robusto.
*   **2. La Tahona (Precio absoluto premium, pero USD/m2 más moderado):** Su promedio de USD 2,553/m2 lo ubica en un rango alto, no en el tope. Esto confirma que el valor de una propiedad en La Tahona está más asociado al tamaño de la construcción que a un sobreprecio extremo por metro, un dato relevante para quien evalúe construir en un lote de la zona en vez de comprar ya edificado.
*   **3. Solymar:** Con 167 propiedades consideradas (más del quíntuple que cualquier otro barrio), Solymar tiene el USD/m2 más bajo entre los barrios "normales" (USD 2,314) y es también el dato estadísticamente más sólido de toda la tabla.
*   **4. El Pinar es la gran excepción:** Tiene el m2 más accesible pero con una brecha inusual entre Media y Mediana. Su promedio de USD 1,666/m2 es notablemente más bajo que el resto, pero llama la atención que la mediana (USD 1,286) esté muy por debajo de la media. Esto es una señal de que unas pocas propiedades más caras están tirando el promedio hacia arriba en una muestra chica (apenas 15 propiedades, la más reducida de todas). Este barrio es el de menor confiabilidad de datos así que esta cifra refuerza la necesidad de leerlo con cautela.

---

### **Pregunta 2. Rentabilidad: venta vs. alquiler (Yield)**

En esta sección buscamos identificar qué barrios ofrecen la mejor relación entre el precio de compra de una vivienda y el ingreso que generaría alquilarla, calculando el **yield bruto anual** (renta anual / precio de venta) por barrio.

Para lograr un análisis robusto y representativo, calculamos bajo dos enfoques estadísticos:

Enfoque de la Media (Promedio): Refleja el rendimiento global del mercado en cada zona.

Enfoque de la Mediana (Percentil 50): Evita la distorsión de valores atípicos (propiedades inusualmente caras o baratas), ofreciendo una visión del comportamiento del "inmueble típico" en el barrio.

**Nota metodológica importante:**
Como el alquiler se cotiza mayormente en pesos uruguayos (excepto en La Tahona, donde el mercado está dolarizado) y la venta se mide en dólares, para poder comparar todos los barrios en una misma moneda convertimos los valores en UYU a USD usando una tasa de referencia de **40 UYU = 1 USD** (cotización aproximada de mercado para Julio de 2026). Esta conversión es una asunción puntual para el análisis de este módulo y no una serie histórica, por lo que debe tenerse en cuenta al momento de interpretar y auditar los resultados.

---


In [ ]:
# =============================================================================
# RENTABILIDAD (YIELD) — VENTA vs. ALQUILER POR BARRIO
# =============================================================================

# Calculamos, para cada barrio, el precio promedio (media) y la mediana de venta
# y de alquiler de viviendas (excluyendo terrenos, que no se alquilan bajo esta lógica).
query_yield = """
WITH precios_base AS (
    SELECT
        ubicacion AS Barrio,

        -- PRECIOS DE VENTA (Siempre en USD)
        -- Precio promedio (Media) de venta de viviendas
        AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                  AND LOWER(moneda) = 'u$s'
                  AND LOWER(operacion) = 'venta'
             THEN precio END) OVER(PARTITION BY ubicacion) AS precio_venta_usd_media,

        -- Precio típico (Mediana) de venta de viviendas
        PERCENTILE_CONT(
            CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                      AND LOWER(moneda) = 'u$s'
                      AND LOWER(operacion) = 'venta'
                 THEN precio END, 0.5
        ) OVER(PARTITION BY ubicacion) AS precio_venta_usd_mediana,


        -- ALQUILERES NATIVOS EN USD (Caso La Tahona)
        -- Precio promedio (Media) de alquiler en USD
        AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                  AND LOWER(moneda) = 'u$s'
                  AND LOWER(operacion) = 'alquiler'
             THEN precio END) OVER(PARTITION BY ubicacion) AS alquiler_usd_media,

        -- Precio típico (Mediana) de alquiler en USD
        PERCENTILE_CONT(
            CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                      AND LOWER(moneda) = 'u$s'
                      AND LOWER(operacion) = 'alquiler'
                 THEN precio END, 0.5
        ) OVER(PARTITION BY ubicacion) AS alquiler_usd_mediana,


        -- ALQUILERES NATIVOS EN UYU (Resto de los barrios)
        -- Precio promedio (Media) de alquiler en pesos uruguayos
        AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                  AND LOWER(moneda) = 'uyu'
                  AND LOWER(operacion) = 'alquiler'
             THEN precio END) OVER(PARTITION BY ubicacion) AS alquiler_uyu_media,

        -- Precio típico (Mediana) de alquiler en pesos uruguayos
        PERCENTILE_CONT(
            CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                      AND LOWER(moneda) = 'uyu'
                      AND LOWER(operacion) = 'alquiler'
                 THEN precio END, 0.5
        ) OVER(PARTITION BY ubicacion) AS alquiler_uyu_mediana

    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
),

precios_unicos_por_barrio AS (
    -- Dado que usamos funciones de ventana, reducimos los duplicados
    -- agrupando para obtener una única fila limpia por cada barrio.
    SELECT
        Barrio,
        ANY_VALUE(precio_venta_usd_media) AS precio_venta_usd_media,
        ANY_VALUE(precio_venta_usd_mediana) AS precio_venta_usd_mediana,
        ANY_VALUE(alquiler_usd_media) AS alquiler_usd_media,
        ANY_VALUE(alquiler_usd_mediana) AS alquiler_usd_mediana,
        ANY_VALUE(alquiler_uyu_media) AS alquiler_uyu_media,
        ANY_VALUE(alquiler_uyu_mediana) AS alquiler_uyu_mediana
    FROM
        precios_base
    GROUP BY
        Barrio
)

# Con los promedios y medianas ya calculados, consolidamos todo en una sola moneda (USD)
# y calculamos el yield bruto anual de cada barrio para ambos enfoques.
SELECT
    Barrio,

    -- Valores de Venta (Media vs. Mediana)
    ROUND(precio_venta_usd_media, 0) AS Precio_Venta_Promedio_USD,
    ROUND(precio_venta_usd_mediana, 0) AS Precio_Venta_Mediano_USD,

    -- Valores de Alquiler Nativo (Media vs. Mediana)
    ROUND(alquiler_usd_media, 0) AS Alquiler_Nativo_USD_Media,
    ROUND(alquiler_usd_mediana, 0) AS Alquiler_Nativo_USD_Mediana,
    ROUND(alquiler_uyu_media, 0) AS Alquiler_Nativo_UYU_Media,
    ROUND(alquiler_uyu_mediana, 0) AS Alquiler_Nativo_UYU_Mediana,

    -- Convertimos los alquileres en pesos a dólares usando una tasa de referencia
    -- de 40 UYU = 1 USD (cotización aproximada de mercado, julio 2026).
    -- ASUNCIÓN METODOLÓGICA: esta tasa es una foto puntual y no una serie
    -- histórica; si el tipo de cambio varía significativamente, este resultado
    -- debería recalcularse.
    ROUND(SAFE_DIVIDE(alquiler_uyu_media, 40), 0) AS Alquiler_UYU_Convertido_a_USD_Media,
    ROUND(SAFE_DIVIDE(alquiler_uyu_mediana, 40), 0) AS Alquiler_UYU_Convertido_a_USD_Mediana,

    -- Unificamos en una sola columna el alquiler mensual en USD:
    -- usamos el valor nativo en dólares si existe (Tahona), y si no,
    -- el valor convertido desde pesos (resto de los barrios).
    ROUND(COALESCE(alquiler_usd_media, SAFE_DIVIDE(alquiler_uyu_media, 40)), 0) AS Alquiler_Consolidado_USD_Media,
    ROUND(COALESCE(alquiler_usd_mediana, SAFE_DIVIDE(alquiler_uyu_mediana, 40)), 0) AS Alquiler_Consolidado_USD_Mediana,

    -- Yield bruto anual basado en la Media (%): (alquiler mensual promedio x 12 meses) / precio de venta promedio x 100.
    ROUND(
        SAFE_DIVIDE(
            COALESCE(alquiler_usd_media, SAFE_DIVIDE(alquiler_uyu_media, 40)) * 12,
            precio_venta_usd_media
        ) * 100, 2
    ) AS Yield_Bruto_Anual_Media_Pct,

    -- Yield bruto anual basado en la Mediana (%): (alquiler mensual mediano x 12 meses) / precio de venta mediano x 100.
    ROUND(
        SAFE_DIVIDE(
            COALESCE(alquiler_usd_mediana, SAFE_DIVIDE(alquiler_uyu_mediana, 40)) * 12,
            precio_venta_usd_mediana
        ) * 100, 2
    ) AS Yield_Bruto_Anual_Mediana_Pct
FROM
    precios_unicos_por_barrio
ORDER BY
    Yield_Bruto_Anual_Mediana_Pct DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_yield = client.query(query_yield).to_dataframe()

# Desplegamos el resultado.
df_yield

,Barrio,Precio_Venta_Promedio_USD,Precio_Venta_Mediano_USD,Alquiler_Nativo_USD_Media,Alquiler_Nativo_USD_Mediana,Alquiler_Nativo_UYU_Media,Alquiler_Nativo_UYU_Mediana,Alquiler_UYU_Convertido_a_USD_Media,Alquiler_UYU_Convertido_a_USD_Mediana,Alquiler_Consolidado_USD_Media,Alquiler_Consolidado_USD_Mediana,Yield_Bruto_Anual_Media_Pct,Yield_Bruto_Anual_Mediana_Pct
0,Tahona,650033.0,590000.0,3514.0,3500.0,NaN,NaN,NaN,NaN,3514.0,3500.0,6.49,7.12
1,Carrasco,312314.0,295000.0,NaN,NaN,43667.0,45000.0,1092.0,1125.0,1092.0,1125.0,4.19,4.58
2,Shangrila,302125.0,305000.0,NaN,NaN,43415.0,39000.0,1085.0,975.0,1085.0,975.0,4.31,3.84
3,Lagomar,294320.0,270000.0,NaN,NaN,39286.0,34000.0,982.0,850.0,982.0,850.0,4.00,3.78
4,Solymar,276716.0,258000.0,NaN,NaN,34435.0,31000.0,861.0,775.0,861.0,775.0,3.73,3.60
5,Pinar,245444.0,252500.0,NaN,NaN,27531.0,27000.0,688.0,675.0,688.0,675.0,3.37,3.21


### **Diagnóstico: rentabilidad por barrio (Yield)**

**La Tahona lidera con yields más altos:** Su yield mediano (7.12%) supera a la media (6.49%) porque la vivienda típica se compra por menos (USD 590,000) de lo que indica el promedio general (USD 650,033), que está inflado por mansiones de lujo. El mercado está dolarizado y dirigido a un perfil corporativo de alto valor.

**Carrasco mejora con la mediana:** Su rentabilidad real típica sube al 4.58% (frente al 4.19% medio) porque el precio de entrada del "inmueble estándar" (USD 295,000) es inferior al promedio del barrio.

**Shangrilá y Lagomar a la baja:** Sus medianas caen al 3.84% y 3.78% respectivamente, esto demuestra que los promedios de alquiler están inflados por unas pocas casas inusualmente caras; el negocio típico de renta residencial rinde menos de lo que sugiere la media.

**Solymar y El Pinar registran los menores retornos:** Se estabilizan en el entorno del 3.60% y 3.21% (mediana). Sin embargo, Solymar compensa su menor yield con una barrera de entrada más accesible y la mayor liquidez de mercado de la zona.


### **Conclusión de Negocio**
* **Para maximizar flujo de caja:** La Tahona es la opción óptima ofreciendo un yield típico real del 7.12% con rentas en dólares.

* **Para menor capital de entrada y rápida salida (liquidez):** Solymar se mantiene como la alternativa más segura y de fácil reventa asumiendo un yield típico del 3.60% pero con menor riesgo de vacancia.

---

### **Pregunta 3. Terreno vs. vivienda construida (Análisis de Viabilidad de Desarrollo)**
Buscamos identificar en qué barrios es comercialmente más conveniente adquirir un terreno para construir desde cero o comprar una propiedad ya edificada comparando el precio por m2 de ambas alternativas.

**Incidencia de suelo baja:** Mayor margen de ganancia para el desarrollador lo que incentiva la construcción y autogestión.

**Incidencia de suelo alta:** El margen de desarrollo se contrae por lo que adquirir una propiedad terminada es más eficiente en tiempo, costo y esfuerzo.

In [ ]:
# =============================================================================
# TERRENO VS. VIVIENDA CONSTRUIDA — COMPARACIÓN DE $/M² POR BARRIO
# =============================================================================

# Calculamos el $/m2 de cada terreno individual (misma lógica que en la Pregunta 1),
# y por separado el $/m2 de cada vivienda construida, para después comparar
# ambos promedios (medias) y valores típicos (medianas) por barrio.
query_terreno_vs_construccion = """
WITH m2_terreno_base AS (
    SELECT
        ubicacion AS Barrio,

        -- $/m2 de cada terreno individual en venta, en dólares.
        SAFE_DIVIDE(precio, mts2_terreno) AS precio_m2_terreno
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        LOWER(tipo_inmueble) = 'terreno'
        AND LOWER(moneda) = 'u$s'
        AND LOWER(operacion) = 'venta'
        AND mts2_terreno IS NOT NULL
        AND mts2_terreno > 0
),

m2_terreno AS (
    SELECT
        Barrio,
        precio_m2_terreno,
        -- Calculamos la mediana del m2 usando función analítica (cláusula OVER) requerida por BigQuery
        PERCENTILE_CONT(precio_m2_terreno, 0.5) OVER(PARTITION BY Barrio) AS precio_m2_terreno_mediana
    FROM
        m2_terreno_base
),

m2_edificado_base AS (
    SELECT
        ubicacion AS Barrio,

        -- $/m2 de cada vivienda construida en venta, en dólares.
        SAFE_DIVIDE(precio, mts2_edificado) AS precio_m2_edificado
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        LOWER(tipo_inmueble) != 'terreno'
        AND LOWER(moneda) = 'u$s'
        AND LOWER(operacion) = 'venta'
        AND mts2_edificado IS NOT NULL
        AND mts2_edificado > 0
),

m2_edificado AS (
    SELECT
        Barrio,
        precio_m2_edificado,
        -- Calculamos la mediana del m2 usando función analítica (cláusula OVER) requerida por BigQuery
        PERCENTILE_CONT(precio_m2_edificado, 0.5) OVER(PARTITION BY Barrio) AS precio_m2_edificado_mediana
    FROM
        m2_edificado_base
),

-- Agregamos por barrio el promedio (media) y la mediana de $/m2 de terreno.
resumen_terreno AS (
    SELECT
        Barrio,
        COUNT(*) AS Cantidad_Terrenos,
        ROUND(AVG(precio_m2_terreno), 0) AS Precio_x_M2_Terreno_Media_USD,
        ROUND(ANY_VALUE(precio_m2_terreno_mediana), 0) AS Precio_x_M2_Terreno_Mediana_USD
    FROM
        m2_terreno
    GROUP BY
        Barrio
),

-- Agregamos por barrio el promedio (media) y la mediana de $/m2 edificado.
resumen_edificado AS (
    SELECT
        Barrio,
        COUNT(*) AS Cantidad_Viviendas,
        ROUND(AVG(precio_m2_edificado), 0) AS Precio_x_M2_Edificado_Media_USD,
        ROUND(ANY_VALUE(precio_m2_edificado_mediana), 0) AS Precio_x_M2_Edificado_Mediana_USD
    FROM
        m2_edificado
    GROUP BY
        Barrio
)

# Unimos ambos resúmenes por barrio (FULL OUTER JOIN por si algún barrio
# no tuviera datos de una de las dos categorías), y calculamos el ratio
# que compara ambos valores bajo el enfoque de la Media y de la Mediana.
SELECT
    COALESCE(t.Barrio, e.Barrio) AS Barrio,
    t.Cantidad_Terrenos,
    t.Precio_x_M2_Terreno_Media_USD,
    t.Precio_x_M2_Terreno_Mediana_USD,
    e.Cantidad_Viviendas,
    e.Precio_x_M2_Edificado_Media_USD,
    e.Precio_x_M2_Edificado_Mediana_USD,

    -- Ratio basado en la Media: qué porcentaje del valor del m2 edificado promedio representa el m2 de terreno promedio.
    ROUND(SAFE_DIVIDE(t.Precio_x_M2_Terreno_Media_USD, e.Precio_x_M2_Edificado_Media_USD) * 100, 1) AS Terreno_Como_Pct_del_Edificado_Media,

    -- Ratio basado en la Mediana: qué porcentaje del valor del m2 edificado típico representa el m2 de terreno típico.
    -- Un porcentaje BAJO indica que el terreno es "barato" en relación a lo construido
    -- (más margen para desarrollar). Un porcentaje ALTO indica que el terreno ya
    -- casi vale lo mismo que comprar construido (menos incentivo a desarrollar).
    ROUND(SAFE_DIVIDE(t.Precio_x_M2_Terreno_Mediana_USD, e.Precio_x_M2_Edificado_Mediana_USD) * 100, 1) AS Terreno_Como_Pct_del_Edificado_Mediana
FROM
    resumen_terreno t
FULL OUTER JOIN
    resumen_edificado e
ON
    t.Barrio = e.Barrio
ORDER BY
    Terreno_Como_Pct_del_Edificado_Mediana ASC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_terreno_vs_construccion = client.query(query_terreno_vs_construccion).to_dataframe()

# Desplegamos el resultado.
df_terreno_vs_construccion

,Barrio,Cantidad_Terrenos,Precio_x_M2_Terreno_Media_USD,Precio_x_M2_Terreno_Mediana_USD,Cantidad_Viviendas,Precio_x_M2_Edificado_Media_USD,Precio_x_M2_Edificado_Mediana_USD,Terreno_Como_Pct_del_Edificado_Media,Terreno_Como_Pct_del_Edificado_Mediana
0,Carrasco,20,256.0,205.0,19,2724.0,3000.0,9.4,6.8
1,Tahona,25,193.0,211.0,28,2553.0,2722.0,7.6,7.8
2,Solymar,56,271.0,225.0,167,2314.0,2396.0,11.7,9.4
3,Lagomar,3,323.0,347.0,25,2641.0,2846.0,12.2,12.2
4,Shangrila,5,481.0,396.0,30,2465.0,2750.0,19.5,14.4
5,Pinar,6,226.0,198.0,15,1666.0,1286.0,13.6,15.4


### **Diagnóstico: Terreno vs. vivienda construida**

**Baja incidencia de la tierra en general:** En toda la zona, el valor del suelo representa una fracción minoritaria de la propiedad terminada (máximo 15.4%), esto asegura que el desarrollo desde cero tiene margen de ganancia en todos los barrios.

**Carrasco es el líder real en márgenes:** El promedio simple sugería un ratio del 9.4% pero la mediana revela que el lote típico cuesta USD 205/m2 frente a un edificado de USD 3,000/m2. Esto baja la incidencia real al 6.8% ofreciendo el mayor beneficio potencial de la muestra.

**La Tahona se mantiene premium y eficiente:** Con un ratio típico del 7.8% muestra una excelente consistencia entre media y mediana. Sumado a su alto yield de alquiler, es el barrio ideal para un modelo de construir para rentar.

**Solymar destaca por volumen y seguridad:** Mantiene un ratio muy saludable del 9.4%. Al poseer las muestras más robustas (56 terrenos y 167 casas) ofrece el dato estadísticamente más confiable para desarrollos rápidos.

**Márgenes más ajustados en Shangrilá y El Pinar:** En Shangrilá la escasez de suelo presiona la incidencia típica al 14.4%. En El Pinar, aunque la tierra es barata (USD 198/m2) el bajo valor del m2 edificado eleva la incidencia al 15.4%, reduciendo el margen porcentual de ganancia.

**Calidad de datos:** Las muestras de terrenos en Lagomar (3) y Shangrilá (5) son reducidas; estas cifras deben tomarse como tendencias orientativas.

**Nota agregada en la fase empresarial:** *esta métrica se recalculó con una aproximación distinta para el panel de control (precio por m² en vez de tamaños típicos).*

*Ver ficha técnica del panel para el detalle.*

---

### **Pregunta 4. Mix ideal de portafolio (Diversificación por barrio)**

Nuestro dataset no tiene una métrica real de demanda (no sabemos cuántas personas vieron o consultaron cada aviso, solo que fue publicado) por eso, en vez de hablar de "oferta y demanda" en sentido estricto vamos a construir el análisis de portafolio con las señales que sí tenemos del lado de la oferta:

*   **Volumen de stock por barrio** (como proxy de liquidez: más publicaciones = mercado más activo).
*   **Precio promedio y segmento de mercado** (accesible / medio / premium) para armar una cartera diversificada por nivel de precio.
*   **Volatilidad de precio dentro de cada barrio** (desvío estándar), como indicador de riesgo: un barrio con precios muy dispersos es menos predecible que uno homogéneo.

Con esto buscamos responder: si una inmobiliaria quisiera armar una cartera balanceada entre distintos barrios, ¿cómo se comparan en volumen, accesibilidad y riesgo de precio?



In [ ]:
# =============================================================================
# VOLUMEN, SEGMENTO Y VOLATILIDAD POR BARRIO
# =============================================================================

# Calculamos, para cada barrio, el volumen de stock, el precio promedio,
# la volatilidad de precio y la clasificación de segmento (accesible/medio/premium).
query_mix_portafolio = """
WITH metricas_barrio AS (
    SELECT
        ubicacion AS Barrio,

        -- Volumen total de stock del barrio (todas las propiedades, cualquier tipo).
        COUNT(*) AS Total_Propiedades,

        -- Precio promedio de venta de viviendas construidas, en USD.
        AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                  AND LOWER(moneda) = 'u$s'
                  AND LOWER(operacion) = 'venta'
             THEN precio END) AS Precio_Promedio_Vivienda_USD,

        -- Desvío estándar del precio de venta de viviendas: mide qué tan
        -- disperso (volátil) es el precio dentro del mismo barrio.
        -- Un desvío alto indica un barrio con oferta muy heterogénea (riesgoso
        -- de predecir); un desvío bajo indica un mercado más uniforme.
        STDDEV_SAMP(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                          AND LOWER(moneda) = 'u$s'
                          AND LOWER(operacion) = 'venta'
                     THEN precio END) AS Volatilidad_Precio_USD
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    GROUP BY
        ubicacion
)

# Con las métricas base calculadas, clasificamos los barrios en segmentos
# de precio (Accesible / Medio / Premium) usando NTILE, que divide los 6
# barrios en 3 grupos iguales según su precio promedio.
SELECT
    Barrio,
    Total_Propiedades,

    -- Participación del barrio sobre el stock total de la Costa (%).
    ROUND(SAFE_DIVIDE(Total_Propiedades, SUM(Total_Propiedades) OVER()) * 100, 1) AS Participacion_Stock_Pct,

    ROUND(Precio_Promedio_Vivienda_USD, 0) AS Precio_Promedio_Vivienda_USD,
    ROUND(Volatilidad_Precio_USD, 0) AS Volatilidad_Precio_USD,

    -- Coeficiente de variación: desvío estándar como % del promedio.
    -- Permite comparar la volatilidad entre barrios de forma relativa,
    -- ya que un desvío de USD 50,000 no significa lo mismo en un barrio
    -- de USD 250,000 promedio que en uno de USD 650,000 promedio.
    ROUND(SAFE_DIVIDE(Volatilidad_Precio_USD, Precio_Promedio_Vivienda_USD) * 100, 1) AS Coef_Variacion_Pct,

    -- Clasificación de segmento según el precio promedio del barrio,
    -- comparado contra el resto de la Costa (NTILE los divide en 3 tercios).
    CASE NTILE(3) OVER (ORDER BY Precio_Promedio_Vivienda_USD)
        WHEN 1 THEN 'Accesible'
        WHEN 2 THEN 'Medio'
        WHEN 3 THEN 'Premium'
    END AS Segmento_Precio
FROM
    metricas_barrio
ORDER BY
    Precio_Promedio_Vivienda_USD DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_mix_portafolio = client.query(query_mix_portafolio).to_dataframe()

# Desplegamos el resultado.
df_mix_portafolio

,Barrio,Total_Propiedades,Participacion_Stock_Pct,Precio_Promedio_Vivienda_USD,Volatilidad_Precio_USD,Coef_Variacion_Pct,Segmento_Precio
0,Tahona,62,11.2,650033.0,189584.0,29.2,Premium
1,Carrasco,44,7.9,312314.0,226441.0,72.5,Premium
2,Shangrila,63,11.4,302125.0,104298.0,34.5,Medio
3,Lagomar,35,6.3,294320.0,153103.0,52.0,Medio
4,Solymar,314,56.6,276716.0,116758.0,42.2,Accesible
5,Pinar,37,6.7,245444.0,165094.0,67.3,Accesible


### **Diagnóstico: Mix de portafolio por barrio**

*   **1. Solymar concentra más de la mitad del stock total (56.6%):** Es el barrio más líquido y de mayor rotación esperada y funciona como la "base" natural de cualquier cartera diversificada (el segmento de mayor volumen de transacciones potenciales).
*   **2. La Tahona es el único Premium con volatilidad contenida:** Con un coeficiente de variación de 29.2% (el más bajo de toda la tabla) a pesar de ser el barrio más caro sus precios son los más predecibles y homogéneos, una característica atractiva para un inversor que busca un segmento premium pero sin sorpresas de precio.
*   **3. Carrasco es una señal de alerta pese a estar en el segmento Premium:** Tiene el coeficiente de variación más alto de la tabla (72.5%), aunque su precio promedio es alto la dispersión de precios dentro del barrio es enorme. Esto sugiere una oferta muy heterogénea (probablemente mezclando propiedades muy distintas entre sí), lo que lo convierte en el barrio de mayor riesgo de predicción de precio de toda la Costa.
*   **4. El Pinar:** Está clasificado como el segmento más accesible por precio promedio pero tiene el segundo coeficiente de variación más alto (67.3%), esto contradice la intuición de que un barrio más barato es automáticamente más predecible, acá conviven propiedades de precio muy dispar dentro de la misma categoría "accesible".
*   **5. Shangrilá es el barrio más estable del segmento medio:** Con el coeficiente de variación más bajo después de La Tahona (34.5%) combina precio medio con buena previsibilidad, un perfil interesante para diversificar sin asumir mucho riesgo.

---

### **Pregunta 5. Dinámica de monedas: USD vs. UYU por tipo de operación**

Buscamos cuantificar qué proporción del mercado se cotiza en cada moneda y si ese patrón cambia según se trate de venta o alquiler. Ya vimos indicios de esto en los módulos anteriores (el alquiler tiende a pesos y la venta tiende a dólares con La Tahona como excepción), pero acá lo medimos a nivel global de todo el Dataset no solo por barrio.

In [ ]:
# =============================================================================
# DISTRIBUCIÓN DE MONEDA (USD vs. UYU) SEGÚN TIPO DE OPERACIÓN
# =============================================================================

# Contamos cuántas publicaciones hay de cada combinación moneda + operación,
# y calculamos qué porcentaje representa cada moneda dentro de cada tipo
# de operación (venta y alquiler por separado).
query_dinamica_monedas = """
WITH conteo_operacion_moneda AS (
    SELECT
        operacion AS Operacion,
        moneda AS Moneda,
        COUNT(*) AS Cantidad_Publicaciones
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        -- Excluimos terrenos para que la lectura de "venta" y "alquiler"
        -- se enfoque en el comportamiento general del mercado residencial,
        -- ya que los terrenos prácticamente no se alquilan.
        LOWER(tipo_inmueble) != 'terreno'
    GROUP BY
        Operacion, Moneda
)

# Calculamos el porcentaje que representa cada moneda DENTRO de cada
# tipo de operación, usando una función de ventana para sumar el total
# de cada grupo (venta / alquiler) sin necesidad de una subconsulta aparte.
SELECT
    Operacion,
    Moneda,
    Cantidad_Publicaciones,
    SUM(Cantidad_Publicaciones) OVER (PARTITION BY Operacion) AS Total_Publicaciones_Operacion,

    -- Porcentaje de esta moneda sobre el total de esa operación específica.
    ROUND(
        SAFE_DIVIDE(
            Cantidad_Publicaciones,
            SUM(Cantidad_Publicaciones) OVER (PARTITION BY Operacion)
        ) * 100, 1
    ) AS Porcentaje_de_la_Operacion
FROM
    conteo_operacion_moneda
ORDER BY
    Operacion, Porcentaje_de_la_Operacion DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_dinamica_monedas = client.query(query_dinamica_monedas).to_dataframe()

# Desplegamos el resultado.
df_dinamica_monedas

,Operacion,Moneda,Cantidad_Publicaciones,Total_Publicaciones_Operacion,Porcentaje_de_la_Operacion
0,Alquiler,UYU,124,131,94.7
1,Alquiler,U$S,7,131,5.3
2,Venta,U$S,309,309,100.0


### **Diagnóstico: Dinámica de monedas por operación**

*   **1. La venta está 100% dolarizada:** Las 309 publicaciones de venta del Dataset (excluyendo terrenos) están cotizadas en USD, sin una sola excepción en pesos. Esto confirma algo característico del mercado inmobiliario uruguayo: el dólar es la moneda de referencia indiscutida para transacciones de compra/venta independientemente del barrio o segmento.
*   **2. El alquiler es mayormente en pesos (94.7%):** Solo 7 de las 131 publicaciones de alquiler (5.3%) están en dólares, y sabemos (por el análisis de barrios del Módulo 1 y la pregunta 2) que esa porción minoritaria en USD corresponde casi enteramente a La Tahona, el único barrio donde el alquiler se dolariza.
*   **3. La brecha entre venta y alquiler revela dos lógicas económicas distintas:** La compra/venta se rige por el valor del activo (protegido en moneda dura, dólares) mientras que el alquiler (un gasto corriente y recurrente ligado al ingreso mensual del inquilino) se maneja naturalmente en la moneda con la que la gente cobra su sueldo: pesos uruguayos. Es un patrón consistente con la economía uruguaya en general, no exclusivo de Ciudad de la Costa.
*   **4. La excepción de La Tahona es una señal de segmento:** El 5.3% de alquileres en dólares no está distribuido al azar entre barrios, se concentra en el segmento premium reforzando la idea de que la dolarización del alquiler es un marcador de status socioeconómico, no una práctica generalizada del mercado.

Cualquier análisis de precios de **venta** en este Dataset puede tratarse como una serie homogénea en USD sin necesidad de conversión.

Para alquiler, cualquier comparación entre barrios debe considerar la moneda (como ya hicimos en la pregunta 2) porque mezclar UYU y USD sin ajustar distorsiona los promedios.

Dado que la dolarización se concentra casi enteramente en un solo barrio no calculamos una correlación formal por ser estadísticamente poco significativa, el patrón se explica mejor de forma cualitativa: **el alquiler se dolariza en el segmento premium y no en el resto de ubicaciones.**

---

### **Pregunta 6. ¿El tamaño de la propiedad afecta el precio por m2?**
Buscamos entender si las propiedades más grandes tienen un valor por m2 más bajo (economía de escala: comprar más metros sale proporcionalmente más barato) o más alto (prima por exclusividad: las propiedades grandes son escasas y eso las encarece por metro). La respuesta le sirve directamente a un desarrollador para decidir si construir unidades chicas o grandes maximiza el retorno por metro cuadrado invertido.

In [ ]:
# =============================================================================
# CORRELACIÓN ENTRE TAMAÑO (M² EDIFICADO) Y PRECIO POR M²
# =============================================================================

# Calculamos, para cada vivienda individual, sus metros edificados y su
# precio por m2, para después medir si existe relación entre ambas variables.
query_tamano_vs_precio_m2 = """
WITH propiedades_individuales AS (
    SELECT
        ubicacion AS Barrio,
        mts2_edificado,

        -- Precio por m2 de esta propiedad puntual.
        SAFE_DIVIDE(precio, mts2_edificado) AS precio_m2
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        LOWER(tipo_inmueble) != 'terreno'
        AND LOWER(moneda) = 'u$s'
        AND LOWER(operacion) = 'venta'
        AND mts2_edificado IS NOT NULL
        AND mts2_edificado > 0
)

# Calculamos la correlación de Pearson entre tamaño y precio por m2,
# tanto a nivel global (toda la Costa) como desglosado por barrio,
# para ver si el patrón es consistente en todas las zonas o varía.
SELECT
    Barrio,
    COUNT(*) AS Cantidad_Propiedades,
    ROUND(AVG(mts2_edificado), 0) AS M2_Promedio,
    ROUND(AVG(precio_m2), 0) AS Precio_x_M2_Promedio,

    -- Correlación de Pearson entre m2 edificados y precio por m2, por barrio.
    -- Negativa = a mayor tamaño, menor precio por m2 (economía de escala).
    -- Positiva = a mayor tamaño, mayor precio por m2 (prima por exclusividad).
    ROUND(CORR(mts2_edificado, precio_m2), 3) AS Correlacion_Tamano_vs_PrecioM2
FROM
    propiedades_individuales
GROUP BY
    Barrio
ORDER BY
    Cantidad_Propiedades DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_tamano_vs_precio_m2 = client.query(query_tamano_vs_precio_m2).to_dataframe()

# Desplegamos el resultado.
df_tamano_vs_precio_m2

,Barrio,Cantidad_Propiedades,M2_Promedio,Precio_x_M2_Promedio,Correlacion_Tamano_vs_PrecioM2
0,Solymar,167,135.0,2314.0,-0.636
1,Shangrila,30,131.0,2465.0,-0.591
2,Tahona,28,298.0,2553.0,-0.847
3,Lagomar,25,124.0,2641.0,-0.537
4,Carrasco,19,124.0,2724.0,-0.066
5,Pinar,15,169.0,1666.0,-0.363


### **Diagnóstico: Tamaño de la propiedad vs. precio por m2**
**1. El patrón general es claro y consistente: a mayor tamaño, menor precio por m2:** Cinco de los seis barrios muestran correlación negativa confirmando una economía de escala real en el mercado de Ciudad de la Costa: las propiedades más grandes tienden a costar proporcionalmente menos por metro cuadrado que las chicas.

**2. La Tahona tiene la correlación negativa más fuerte de toda la tabla (-0.847):** es una relación casi lineal, cuanto más grande la propiedad más cae el valor por m2.  Para un desarrollador esto es una señal de negocio concreta: en La Tahona, construir unidades más compactas o medianas es lo que realmente maximiza el retorno por metro invertido.

**3. Solymar y Shangrilá muestran correlaciones negativas moderadas/fuertes (-0.636 y -0.591):** consistentes con el patrón general, el mismo fenómeno de economía de escala aunque menos extremo que en La Tahona.

**4. Carrasco es la excepción notable:** correlación prácticamente nula (-0.066). A diferencia de todos los demás barrios en Carrasco el tamaño de la propiedad casi no influye en el precio por metro cuadrado. Esto es coherente con lo que ya vimos en la Pregunta 4: Carrasco tiene el coeficiente de variación de precios más alto de toda la Costa (72.5%), es decir, una oferta muy heterogénea donde el precio parece depender de otros factores (ubicación específica, terminaciones, cercanía al Aeropuerto) más que del tamaño en sí.

**5. El Pinar muestra la correlación más débil después de Carrasco (-0.363):** con una muestra chica (15 propiedades) coherente con la **advertencia de baja confiabilidad** que ya vimos sobre este barrio desde el Módulo 1.


---
## **Cierre del Módulo:**
En este módulo aplicamos la infraestructura de datos del Módulo 1 para responder seis preguntas de negocio mediante SQL, validando cada resultado con su tamaño de muestra correspondiente.

## **Próximo Módulo:**
Geografía comparativa.

---
Análisis realizado en Julio/2026 con fines académicos.
--